In [ ]:
# Dataset download
# You can use the IBM Telco Customer Churn dataset:
# - Download Telco Customer Churn CSV from GitHub: https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv?utm_source=chatgpt.com
# - IBM Telco Customer Churn project: https://github.com/IBM/telco-customer-churn-on-icp4d?utm_source=chatgpt.com





# ============================================================
# EXPERIMENT NO. 3
# DATA CLEANING AND PREPARATION
# Dataset: Telecom_Customer_Churn.csv
# ============================================================

# STEP 1: Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from google.colab import files

# STEP 2: Upload the dataset

print("Upload Telecom_Customer_Churn.csv")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

# STEP 3: Load the dataset

df = pd.read_csv(file_name)

print("\n========== ORIGINAL DATASET ==========")
print(df.head())

# STEP 4: Explore the dataset

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nStatistical Description:")
print(df.describe(include="all"))

# STEP 5: Clean column names

# Remove extra spaces and standardize column names.

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

print("\nCleaned Column Names:")
print(df.columns.tolist())

# STEP 6: Check missing values

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

print("\nMissing Values Percentage:")
print((df.isnull().mean() * 100).round(2))

# STEP 7: Remove duplicate records

print("\nNumber of Duplicate Rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

# STEP 8: Standardize categorical values

categorical_columns = df.select_dtypes(
    include=["object"]
).columns

for column in categorical_columns:
    df[column] = df[column].astype("string").str.strip()

    # Replace blank strings with missing values.
    df[column] = df[column].replace("", pd.NA)

# Standardize common Yes/No values.

for column in categorical_columns:
    df[column] = df[column].replace({
        "yes": "Yes",
        "YES": "Yes",
        "no": "No",
        "NO": "No"
    })

# STEP 9: Convert numeric columns into proper data types

# In some telecom datasets, TotalCharges is stored as text.
# Convert it into a numeric column.

if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"],
        errors="coerce"
    )

# STEP 10: Handle missing values

# Fill numerical missing values with median.
numeric_columns = df.select_dtypes(
    include=np.number
).columns

for column in numeric_columns:
    df[column] = df[column].fillna(
        df[column].median()
    )

# Fill categorical missing values with mode.
categorical_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in categorical_columns:
    mode_value = df[column].mode(dropna=True)

    if not mode_value.empty:
        df[column] = df[column].fillna(mode_value[0])
    else:
        df[column] = df[column].fillna("Unknown")

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

# STEP 11: Identify outliers using IQR method

numeric_columns = df.select_dtypes(
    include=np.number
).columns

print("\n========== OUTLIER DETECTION ==========")

for column in numeric_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_limit) |
        (df[column] > upper_limit)
    ]

    print(column, ":", len(outliers), "outliers")

# STEP 12: Handle outliers using capping
# Values outside the IQR limits are capped at the limits.
# This retains the records instead of deleting them.

for column in numeric_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    df[column] = df[column].clip(
        lower=lower_limit,
        upper=upper_limit
    )

# STEP 13: Feature Engineering

# Create new features from existing columns.

if "tenure" in df.columns:

    # Group customers based on their tenure.
    df["Tenure_Group"] = pd.cut(
        df["tenure"],
        bins=[-1, 12, 36, 72, float("inf")],
        labels=[
            "New Customer",
            "Regular Customer",
            "Loyal Customer",
            "Long-Term Customer"
        ]
    )

if "MonthlyCharges" in df.columns and "tenure" in df.columns:

    # Estimate total monthly charges over customer tenure.
    df["Estimated_Total_Charges"] = (
        df["MonthlyCharges"] * df["tenure"]
    )

print("\n========== FEATURE ENGINEERING ==========")
print(df.head())

# STEP 14: Visualize customer churn distribution

if "Churn" in df.columns:

    plt.figure(figsize=(6, 4))

    sns.countplot(
        data=df,
        x="Churn"
    )

    plt.title("Customer Churn Distribution")
    plt.xlabel("Churn")
    plt.ylabel("Number of Customers")
    plt.show()

# STEP 15: Visualize numerical data using box plots

numeric_columns = df.select_dtypes(
    include=np.number
).columns

if len(numeric_columns) > 0:

    df[numeric_columns].plot(
        kind="box",
        subplots=True,
        layout=(
            (len(numeric_columns) + 2) // 3,
            3
        ),
        figsize=(12, 8),
        sharex=False,
        sharey=False
    )

    plt.suptitle("Box Plots of Numerical Features")
    plt.tight_layout()
    plt.show()

# STEP 16: Encode categorical variables
# Convert text values into numerical values.

df_encoded = df.copy()

categorical_columns = df_encoded.select_dtypes(
    include=["object", "string", "category"]
).columns

for column in categorical_columns:

    encoder = LabelEncoder()

    df_encoded[column] = encoder.fit_transform(
        df_encoded[column].astype(str)
    )

print("\n========== ENCODED DATASET ==========")
print(df_encoded.head())

# STEP 17: Normalize numerical data using StandardScaler

numeric_columns = df_encoded.select_dtypes(
    include=np.number
).columns.tolist()

# Exclude the target column from scaling.
if "Churn" in numeric_columns:
    numeric_columns.remove("Churn")

scaler = StandardScaler()

df_encoded[numeric_columns] = scaler.fit_transform(
    df_encoded[numeric_columns]
)

print("\n========== SCALED DATASET ==========")
print(df_encoded.head())

# STEP 18: Split data into training and testing sets

if "Churn" in df.columns:

    X = df_encoded.drop(columns=["Churn"])
    y = df_encoded["Churn"]

    # Convert target labels into 0 and 1.
    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(y.astype(str))

    # Use stratification when both classes have enough records.
    stratify_value = y if len(np.unique(y)) > 1 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=stratify_value
    )

    print("\n========== TRAIN TEST SPLIT ==========")
    print("Training Features:", X_train.shape)
    print("Testing Features:", X_test.shape)
    print("Training Target:", y_train.shape)
    print("Testing Target:", y_test.shape)

else:
    print("\nChurn target column not found.")
    print("Data cleaning and preparation completed.")
    X_train = df_encoded
    X_test = None
    y_train = None
    y_test = None

# STEP 19: Export cleaned datasets

df.to_csv("cleaned_telecom_data.csv", index=False)

df_encoded.to_csv(
    "prepared_telecom_data.csv",
    index=False
)

print("\n========== EXPERIMENT COMPLETED ==========")
print("Cleaned dataset saved successfully.")
print("Prepared numerical dataset saved successfully.")

# STEP 20: Download the cleaned dataset

files.download("cleaned_telecom_data.csv")


Upload Telecom_Customer_Churn.csv
